# TPCRP Colab Runner

This notebook runs the current TPCRP CIFAR-10 implementation in Google Colab.

Before running, switch Colab to `Runtime -> Change runtime type -> GPU`.

## Option A: clone from GitHub

Set `REPO_URL` to your GitHub repo URL and run the cell.

In [4]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = "https://github.com/KasimM05/Kasim-5CCSAMLF-CW2"
REPO_DIR = Path("/content/Kasim-5CCSAMLF-CW2")

if REPO_URL:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    print(f"Cloned into {REPO_DIR}")
else:
    print("Set REPO_URL above, or use the zip upload option below.")

Set REPO_URL above, or use the zip upload option below.


## Option B: upload a zip

Zip the repo locally, upload it here, and this cell will unpack it under `/content/`.

If you already cloned from GitHub above, skip this cell.

In [5]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded), None)
if not zip_name:
    raise ValueError("No zip file was uploaded.")

extract_root = Path("/content")
with zipfile.ZipFile(zip_name, "r") as archive:
    archive.extractall(extract_root)

candidate_dirs = [path for path in extract_root.iterdir() if path.is_dir() and path.name.startswith("Kasim-5CCSAMLF-CW2")]
if not candidate_dirs:
    raise FileNotFoundError("Could not find the extracted project directory.")

REPO_DIR = candidate_dirs[0]
print(f"Using extracted repo at {REPO_DIR}")

KeyboardInterrupt: 

## Enter the repo and install dependencies

In [3]:
%cd /content/Kasim-5CCSAMLF-CW2
%pip install -r requirements.txt

[Errno 2] No such file or directory: '/content/Kasim-5CCSAMLF-CW2'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## Check the runtime

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

## Fast smoke test

This verifies the full pipeline on a small active pool.

In [ ]:
!python train.py --debug-subset 256 --query-size 10 --rounds 1 --simclr-epochs 1 --classifier-epochs 1 --simclr-batch-size 128 --num-workers 2

## Larger debug run

This is a better intermediate check before the full CIFAR-10 pool.

In [ ]:
!python train.py --debug-subset 1024 --query-size 10 --rounds 1 --simclr-epochs 3 --classifier-epochs 5 --simclr-batch-size 128 --classifier-batch-size 128 --num-workers 2

## Full CIFAR-10 pool run

This is the important validation run for the coursework implementation. Start with lighter settings first, then scale up if needed.

In [ ]:
!python train.py --query-size 10 --rounds 1 --simclr-epochs 3 --classifier-epochs 5 --simclr-batch-size 128 --classifier-batch-size 128 --num-workers 2

## Inspect saved outputs

In [ ]:
from pathlib import Path
import json
import pandas as pd

output_dir = Path("outputs")
summary_path = output_dir / "run_summary.json"
metrics_path = output_dir / "round_01_metrics.csv"

if summary_path.exists():
    with summary_path.open("r", encoding="utf-8") as handle:
        summary = json.load(handle)
    print(json.dumps(summary, indent=2))
else:
    print("run_summary.json not found")

if metrics_path.exists():
    display(pd.read_csv(metrics_path))
else:
    print("round_01_metrics.csv not found")

## Download result files

In [ ]:
from google.colab import files
from pathlib import Path

for path in [Path("outputs/run_summary.json"), Path("outputs/round_01_metrics.csv")]:
    if path.exists():
        files.download(str(path))